# 02 — Working with LLMs

`🔴 Advanced`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/07_AI_Engineering/02_working_with_llms.ipynb)

## 📌 What is it?
Using official SDKs (OpenAI, Anthropic) to call LLMs — including streaming, multi-turn conversations, token counting, and error handling.

## 🧠 Why AI Engineers need this
This is the daily bread of AI engineering. Mastering the SDK, understanding message structure, and handling edge cases is critical for every LLM application.

In [ ]:
# ── OPENAI SDK ──
# pip install openai

openai_code = '''
from openai import OpenAI
import os

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ── BASIC COMPLETION ──
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful Python expert."},
        {"role": "user", "content": "What is the GIL in Python?"}
    ],
    temperature=0.7,
    max_tokens=500
)
print(response.choices[0].message.content)
print(f"Tokens used: {response.usage.total_tokens}")

# ── STREAMING ──
stream = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Write a haiku about Python"}],
    stream=True
)
for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()
'''
print("OpenAI SDK pattern:")
print(openai_code)

In [ ]:
# ── ANTHROPIC SDK ──
# pip install anthropic

anthropic_code = '''
import anthropic
import os

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# ── BASIC MESSAGE ──
message = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=1024,
    system="You are an expert AI engineer.",
    messages=[
        {"role": "user", "content": "Explain RAG in 3 sentences."}
    ]
)
print(message.content[0].text)
print(f"Input tokens:  {message.usage.input_tokens}")
print(f"Output tokens: {message.usage.output_tokens}")
print(f"Stop reason:   {message.stop_reason}")

# ── STREAMING ──
with client.messages.stream(
    model="claude-3-5-sonnet-20241022",
    max_tokens=500,
    messages=[{"role": "user", "content": "What is Python used for?"}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
    print()
    
    final = stream.get_final_message()
    print(f"Total tokens: {final.usage.input_tokens + final.usage.output_tokens}")
'''
print("Anthropic SDK pattern:")
print(anthropic_code)

In [ ]:
# ── MULTI-TURN CONVERSATION MANAGEMENT ──
from typing import List

class ConversationManager:
    """Manage multi-turn LLM conversations."""
    
    def __init__(self, system_prompt: str, max_history: int = 20):
        self.system_prompt = system_prompt
        self.max_history = max_history
        self.history: List[dict] = []
    
    def add_user(self, content: str):
        self.history.append({"role": "user", "content": content})
        self._trim()
    
    def add_assistant(self, content: str):
        self.history.append({"role": "assistant", "content": content})
    
    def _trim(self):
        if len(self.history) > self.max_history:
            self.history = self.history[-self.max_history:]
    
    def get_messages(self) -> List[dict]:
        return self.history.copy()
    
    def clear(self):
        self.history = []
    
    def __repr__(self):
        return f"ConversationManager(turns={len(self.history)//2})"

# Simulate a conversation
conv = ConversationManager("You are a Python tutor.", max_history=10)

conv.add_user("What is a generator?")
conv.add_assistant("A generator is a function that yields values lazily using the `yield` keyword.")

conv.add_user("Show me an example.")
conv.add_assistant("def count_up(n):\n    for i in range(n):\n        yield i")

print(conv)
print(f"Messages to send: {len(conv.get_messages())}")
for msg in conv.get_messages():
    print(f"  [{msg['role']:9s}] {msg['content'][:60]}")

In [ ]:
# ── TOKEN COUNTING & COST ESTIMATION ──
import tiktoken   # pip install tiktoken

def count_tokens(text: str, model: str = "gpt-4o") -> int:
    """Count tokens using tiktoken (OpenAI's tokenizer)."""
    try:
        enc = tiktoken.encoding_for_model(model)
    except KeyError:
        enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(text))

def estimate_cost(prompt: str, response: str, model: str = "gpt-4o") -> float:
    """Estimate API cost in USD."""
    pricing = {
        "gpt-4o":       {"input": 2.50, "output": 10.00},   # per 1M tokens
        "gpt-4o-mini":  {"input": 0.15, "output":  0.60},
        "gpt-3.5-turbo":{"input": 0.50, "output":  1.50},
    }
    if model not in pricing:
        return 0.0
    p = pricing[model]
    in_tokens  = count_tokens(prompt, model)
    out_tokens = count_tokens(response, model)
    cost = (in_tokens * p["input"] + out_tokens * p["output"]) / 1_000_000
    return cost, in_tokens, out_tokens

system = "You are a helpful AI assistant that writes clear, concise code."
user   = "Write a Python function to chunk text for RAG pipelines."
fake_response = "def chunk_text(text, size=500, overlap=50):\n    chunks = []\n    ..."

cost, in_tok, out_tok = estimate_cost(system + user, fake_response)
print(f"Input tokens:  {in_tok}")
print(f"Output tokens: {out_tok}")
print(f"Estimated cost: ${cost:.6f}")

## 🔗 What's Next?
[03 — Prompt Engineering in Code →](03_prompt_engineering_in_code.ipynb)